## E1a - Data Collection
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 29th, 2026

### About: Today's challenge is open-ended -- you decide what your robot needs to be able to tell apart. This notebook is the same teleoperated data-collection pattern you've used all week (B2, B4, D1), but generalized: you name your own dataset and your own class, instead of them being fixed.
### You'll likely run this notebook's collection steps more than once today -- once per class you need.

### How this works
### Two things you choose: a **dataset name** (a folder that will hold all your classes for this challenge) and a **class name** (a folder inside it, for one specific thing you're teaching your robot to recognize -- e.g. `single`, `stack`).
### Drive around and capture with the **left bumper**, same shutter-button pattern as before. The on-screen button does the same thing, in case driving and shooting at the same time is tricky.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY E1a.1: Collect a Class</span>

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 1. Import required libraries

import os
import ipywidgets as widgets
from IPython.display import display

from jetcam_lite import TraitletCamera

from robot_utils import get_rvr, close_if_exists
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop
from jupyter_utils import register_click_handler
from preview_utils import register_throttled_preview, encode_jpeg
from capture_utils import ensure_directory, save_image, count_images

### STEP 2. Name your dataset and this class. If you're adding another class to a dataset you already started, use the exact same `dataset_name` again.

In [ ]:
##### ----- FEEL FREE TO CHANGE THESE VALUES ----- #####
dataset_name = "my_challenge_dataset"
class_name = "stacked"

### STEP 3. Set up the camera, directory, and collection interface.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

rvr = get_rvr()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg')
# Only a few frames per second are sent to your laptop -- five robots share
# one access point. The images you SAVE are unaffected; see STEP 4.
register_throttled_preview(camera, image_widget, fps=15, max_width=400, quality=50)

data_dir = os.path.join(os.path.expanduser("~"), "Datasets", dataset_name)
class_directory = os.path.join(data_dir, class_name)
ensure_directory(class_directory)

existing_count = count_images(class_directory)
if existing_count > 0:
    print(f"Heads up: '{class_name}' already has {existing_count} images -- "
          f"continuing will add more to the same folder, not start fresh.")

class_count = widgets.IntText(value=existing_count, description=class_name, layout=widgets.Layout(width='200px'))
capture_button = widgets.Button(description=f'Capture {class_name}', button_style='info')

display(image_widget)
display(widgets.HBox([capture_button, class_count]))

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 4. Define what happens on a capture request, and wire up both the
# left bumper and the on-screen button to trigger it.

def capture_image(*args):
    # Encode from camera.value, not image_widget.value: the widget holds the
    # small preview that was sent to your laptop, while camera.value is the
    # full-size frame still on the robot.
    save_image(encode_jpeg(camera.value, quality=95), class_directory, label=class_name)
    class_count.value = count_images(class_directory)

def check_gamepad_captures(gamepad_state):
    if gamepad_state.left_bumper_pressed:
        gamepad_state.left_bumper_pressed = False
        capture_image()

register_click_handler(capture_button, capture_image)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 5. Connect the gamepad and start driving + collecting.

gamepad, gamepad_state = connect_gamepad()

left_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left Stick:',
                                         orientation='vertical', readout=True, readout_format='.2f')
right_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right Stick:',
                                          orientation='vertical', readout=True, readout_format='.2f')
display(widgets.HBox([left_stick_slider, right_stick_slider]))


def on_update(left_value, right_value):
    left_stick_slider.value = left_value
    right_stick_slider.value = right_value
    check_gamepad_captures(gamepad_state)


start_control_loop(rvr, gamepad_state, on_update=on_update)

### Drive around and collect at least 200 images of this class, with real variety -- different angles, distances, lighting, and locations in the field. When you're done, run the cell below to stop.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 6. Stop driving.
stop_control_loop()

### Collecting another class?
### Scroll back up to STEP 2, change `class_name` (keep `dataset_name` the same), then re-run STEP 2 through STEP 6. Everything -- the camera, the robot connection, the gamepad -- is already set up and doesn't need reconnecting.

### Wrapping Up
When you're completely done collecting for now, run the cell below to release the robot's connection.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")